In [218]:
import pandas as pd
import numpy as np

In [ ]:
class FileValidationError(Exception):
    """Incase file not found or doesnt exist!"""
    pass
class FileMissingValueError(FileValidationError):
    """If file exists but contains no value at all!"""
    pass

def load_results(filepath: str):
    if not os.path.exists(filepath):
        raise FileValidationError("File not found!!")  
    
    elif os.path.getsize(filepath) == 0:
        raise FileMissingValueError("File exists but contain no value or data")

    else:
        return pd.read_csv(filepath, na_values=["\\N"])
        
def filter_by_points_range(df: pd.DataFrame, min_points: int, max_points: int) -> pd.DataFrame:
    return df.query("points >= @min_points & points <= @max_points")

def filter_by_grid_and_position(df: pd.DataFrame, grid_positions: list[int], finish_positions: list[str]) -> pd.DataFrame:
    return df[df["grid"].isin(grid_positions) & df["position"].isin(finish_positions)]

df = load_results("../data/pandas_p1/results.csv")

print(df)

print(df.shape)

print(filter_by_points_range(df, min_points=10, max_points=25))

print(filter_by_grid_and_position(df, grid_positions=[1], finish_positions=["1"]))

In [ ]:
import os
import pandas as pd
import numpy as np

class FileValidationError(Exception):
    pass

class FileMissingValueError(FileValidationError):
    pass

def load_results(filepath: str) -> pd.DataFrame:
    if not os.path.exists(filepath):
        raise FileValidationError("File not found!!")  
    elif os.path.getsize(filepath) == 0:
        raise FileMissingValueError("File exists but contains no value or data")
    else:
        return pd.read_csv(filepath)
    
def handle_missing_values(df: pd.DataFrame, strategy: str="drop") -> pd.DataFrame:
    if strategy == "drop":
        df = df.dropna()
    elif strategy == "fill":
        df = df.fillna(df.select_dtypes(include="number").mean())
    else: 
        print("Wrong Message!!")
    return df

def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates()

def fix_dtypes(df: pd.DataFrame, column: str, target_type: str) -> pd.DataFrame:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype(target_type)
    return df

def clean_string_column(df: pd.DataFrame, columns: str) -> pd.DataFrame:
    df[columns] = df[columns].astype(str).str.strip().str.lower()
    return df

if __name__ == "__main__":
    df = load_results("../data/pandas_p1/results.csv")
    df.replace("\\N", np.nan, inplace=True)
    df = remove_duplicates(df)
    df = handle_missing_values(df, "drop")
    df = fix_dtypes(df, "points", "float64")
    # df = clean_string_column(df)


In [39]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/pandas_p1/AB_NYC_2019.csv")

def groupby_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Groups the DataFrame by neighbourhood_group and calculates summary statistics."""
    return df.groupby("neighbourhood_group").agg(
        listing_count=("id", "count"),
        average_price=("price", "mean"),
        average_median=("price", "median"),
        average_availability=("availability_365", "mean")
    ).sort_values("average_price", ascending=False)

groupby_summary(df)

def groupby_multi(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby(["neighbourhood_group", "room_type"]).agg(average_price=("price", "mean")).reset_index()

groupby_multi(df)

In [89]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/pandas_p1/AB_NYC_2019.csv")

def build_borough_lookup() -> pd.DataFrame:
    borough_data = {
    "neighbourhood_group": df["neighbourhood_group"].unique(),
    "borough_population_milltions": [2.6, 1.6, 2.3, 0.5, 1.4],
    "is_manhattan_adjacent": ['True', 'False', 'True', 'False', 'True']
    }

    return borough_data

def merge_listings(df: pd.DataFrame, lookup: pd.DataFrame, how:str) -> pd.DataFrame:
    return df.merge(lookup, on="neighbourhood_group", how=how)

testing = merge_listings(df, borough_info, "left")
testing

testing.shape[0] == df.shape[0]

borough_data = build_borough_lookup()
borough_info = pd.DataFrame(borough_data)
borough_info


borough_info_2 = borough_info.copy()
borough_info_2 = borough_info_2.drop([0, 1])

testing_2 = merge_listings(df, borough_info_2, "left")
testing_2.shape[0] == df.shape[0]






,neighbourhood_group,borough_population_milltions,is_manhattan_adjacent
0,Brooklyn,2.6,True
1,Manhattan,1.6,False
2,Queens,2.3,True
3,Staten Island,0.5,False
4,Bronx,1.4,True


True

True

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,6913795,Nevena's,10716661,Dimitrios,Queens,Ditmars Steinway,40.77228,-73.91345,Entire home/apt,75,3,2,2016-01-31,0.04,1,0
1,34933929,Sonder | 116 John | Cozy Studio + Rooftop,219517861,Sonder (NYC),Manhattan,Financial District,40.70722,-74.00482,Entire home/apt,117,29,0,NaN,NaN,327,345
2,4586880,Vintage Room in Brooklyn,23776693,Bev,Brooklyn,Bedford-Stuyvesant,40.68373,-73.92925,Private room,50,2,100,2019-07-04,3.21,3,260
3,26785267,Comfy and Bright Bedroom in Brooklyn Chinatown,201403610,青明,Brooklyn,Sunset Park,40.64101,-74.01049,Private room,57,2,16,2019-06-05,1.35,5,222
4,34014432,"Cozy , clean and comfortable",125755479,Carla,Queens,Astoria,40.76398,-73.91409,Private room,100,1,0,NaN,NaN,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,35260351,One Bedroom Apartment in a Doorman Building,262078431,Darling,Manhattan,Upper West Side,40.79164,-73.97361,Entire home/apt,169,1,0,NaN,NaN,1,22
196,9013350,Sunny Clinton Hill Apartment,4048695,Deniz,Brooklyn,Clinton Hill,40.68724,-73.96361,Entire home/apt,120,2,28,2019-06-30,0.64,1,294
197,36399640,Queen size bed in long island city,32392762,奕竹,Queens,Long Island City,40.74723,-73.94098,Shared room,45,1,0,NaN,NaN,1,15
198,19721535,Rockaway Beach Oasis,12162815,Annelise,Queens,Rockaway Beach,40.59093,-73.81238,Entire home/apt,140,2,9,2019-06-14,0.38,1,36


,minimum_nights,number_of_reviews
0,1,9
1,1,45
2,3,0
3,1,270
4,10,9
...,...,...
48890,2,0
48891,4,0
48892,10,0
48893,1,0
